Updated model to modify for normalization N and baseline B.

In [534]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [535]:
from src.RealDataReplication import RealDataReplicationDeconvolution
from src.config import load_default_chrom_configs

config, _ = load_default_chrom_configs()
real_deconv1 = RealDataReplicationDeconvolution(config, replicate=1)

Initializing N using config H.
Initializing B using timepoint 0


###### 

In [536]:
# First run to get an initial estimation of F

In [537]:
H, B, G, N = real_deconv1.H, \
    real_deconv1.initial_B, real_deconv1.G, real_deconv1.initial_N

In [541]:
real_deconv1.iterative_deconvolution_updates(total_iterations=1)
F = real_deconv1.F

Iteration 0
0/368 - 00:00:00.073
100/368 - 00:00:01.458
200/368 - 00:00:03.441
300/368 - 00:00:05.241
Iteration completed 00:00:06.169, rn=0.10253732123065967


In [542]:
from src.optimize_H import ParameterOptimizer

In [543]:
model_intervals = config.intervals_wt1
timepoints = config.WT1_TIMEPOINTS
initial_values = {
    'mu0': 26.77,
    'sigma0': 10.9885,
    'lambda_val': 68.19,
    'delta': 8.18,
    'alpha': 22,
    'halted': 0.2245
}
init_params_df = pd.DataFrame(initial_values, index=['value']).T


In [544]:
bounds_dic = {
    'mu0': (20, 100),
    'lambda_val': (50, 80),
    'delta': (0, 24),
    'alpha': (0, 30),
    'sigma0': (1, 15),
    'halted': (0, 1.)
}

bounds_params_df = pd.DataFrame(bounds_dic, index=['min', 'max']).T
params_df = init_params_df.join(bounds_params_df)
params_df


,value,min,max
mu0,26.7700,20.0,100.0
sigma0,10.9885,1.0,15.0
lambda_val,68.1900,50.0,80.0
delta,8.1800,0.0,24.0
alpha,22.0000,0.0,30.0
halted,0.2245,0.0,1.0


In [545]:
start, end = 0, 368

abbreviated_G = G[:, start:end]
abbreviated_F = F[:, start:end]
abbreviated_B = B[:, start:end][start:end, :]

In [550]:
real_deconv1.config.intervals_wt1[2]

[array([-26.7901, -25.5597, -24.3292, -23.0988, -21.8683, -20.6379,
        -19.4075, -18.177 , -16.9466, -15.7161, -14.4857, -13.2553,
        -12.0248, -10.7944,  -9.5639,  -8.3335,  -7.1031,  -5.8726,
         -4.6422,  -3.4117,  -2.1813,  -0.9509,   0.2796]),
 array([ 0.2796,  1.3727,  2.4659,  3.559 ,  4.6521,  5.7453,  6.8384,
         7.9316,  9.0247, 10.1178, 11.211 , 12.3041, 13.3972, 14.4904,
        15.5835, 16.6767, 17.7698, 18.8629, 19.9561, 21.0492, 22.1424,
        23.2355, 24.3286, 25.4218, 26.5149, 27.608 , 28.7012, 29.7943,
        30.8875, 31.9806, 33.0737, 34.1669, 35.26  , 36.3532, 37.4463,
        38.5394, 39.6326, 40.7257, 41.8188, 42.912 , 44.0051, 45.0983,
        46.1914])]

In [546]:
from src.timer import Timer
from src.single_G1_config import calcH

timer = Timer()

num_epochs = 100
num_iterations_N_B = 3

real_deconv1.initial_B = abbreviated_B
real_deconv1.G = abbreviated_G
real_deconv1.F = abbreviated_F

update_params_df = pd.DataFrame()

optimizer = ParameterOptimizer(
    init_params_df=params_df,
    config=config,
    N=N,
    F=abbreviated_F,
    B=abbreviated_B,
    G=abbreviated_G
)

for epoch in range(num_epochs):
    print("Epoch: ", epoch)
    
    optimizer.optimize(maxiter=1000, verbose=True)

    real_deconv1.H = optimizer.current_H
    real_deconv1.iterative_deconvolution_updates(
        total_iterations=num_iterations_N_B, timer=timer)
    timer.print_time()
    
    params_row = pd.DataFrame([optimizer.params_df['value']], index=[epoch])
    params_row['opt_H_loss'] = optimizer.rn
    params_row['F_rn'] = real_deconv1.rn

    update_params_df = pd.concat([update_params_df, params_row])
    
    optimizer.N = real_deconv1.N
    optimizer.B = real_deconv1.B
    optimizer.F = real_deconv1.F
    
    # Setup for the next epoch
    real_deconv1.initial_N = real_deconv1.N
    real_deconv1.initial_B = real_deconv1.B

    print(update_params_df.iloc[-1])


Epoch:  0
Optimization [1]: Current loss: 0.006407626143016293
Optimization [101]: Current loss: 0.006093992007116732
Optimization [201]: Current loss: 0.006088840129319963
Optimization [301]: Current loss: 0.006087288565684893
Optimization [401]: Current loss: 0.006087238741308203
Optimization terminated successfully.
         Current function value: 0.006087
         Iterations: 276
         Function evaluations: 438
Iteration 0
0/368 - 00:01:42.811
100/368 - 00:01:44.121
200/368 - 00:01:45.465
300/368 - 00:01:46.739
Iteration completed 00:01:47.632, rn=0.09552365431596144
Iteration 1
0/368 - 00:01:47.694
100/368 - 00:01:48.987
200/368 - 00:01:50.412
300/368 - 00:01:51.882
Iteration completed 00:01:52.796, rn=0.09134229135918873
Iteration 2
0/368 - 00:01:52.856
100/368 - 00:01:54.402
200/368 - 00:01:55.756
300/368 - 00:01:57.030
Iteration completed 00:01:57.895, rn=0.08907637690405279
00:01:57.895
mu0           27.160331
lambda_val    64.730817
delta         10.788886
sigma0        1

Optimization [1]: Current loss: 0.004713844051335418
Optimization [101]: Current loss: 0.004694972227989903
Optimization [201]: Current loss: 0.004694442874627971
Optimization terminated successfully.
         Current function value: 0.004694
         Iterations: 130
         Function evaluations: 217
Iteration 0
0/368 - 00:12:42.166
100/368 - 00:12:43.406
200/368 - 00:12:44.644
300/368 - 00:12:45.913
Iteration completed 00:12:46.736, rn=0.07508944024812939
Iteration 1
0/368 - 00:12:46.794
100/368 - 00:12:48.014
200/368 - 00:12:49.227
300/368 - 00:12:50.443
Iteration completed 00:12:51.303, rn=0.07516995248168962
Iteration 2
0/368 - 00:12:51.362
100/368 - 00:12:52.585
200/368 - 00:12:53.809
300/368 - 00:12:55.014
Iteration completed 00:12:55.854, rn=0.07503483391263602
00:12:55.855
mu0           25.283581
lambda_val    57.473338
delta          7.566543
sigma0        12.074686
alpha         30.000000
halted         0.000002
opt_H_loss     0.004694
F_rn           0.075035
Name: 8, dtype:

300/368 - 00:19:20.144
Iteration completed 00:19:20.965, rn=0.07452255862548829
Iteration 2
0/368 - 00:19:21.024
100/368 - 00:19:22.237
200/368 - 00:19:23.443
300/368 - 00:19:24.655
Iteration completed 00:19:25.489, rn=0.07449695238773997
00:19:25.489
mu0           23.113385
lambda_val    56.495869
delta          5.451736
sigma0        12.208195
alpha         30.000000
halted         0.000002
opt_H_loss     0.004654
F_rn           0.074497
Name: 16, dtype: float64
Epoch:  17
Optimization [1]: Current loss: 0.004656059524233748
Optimization terminated successfully.
         Current function value: 0.004655
         Iterations: 53
         Function evaluations: 99
Iteration 0
0/368 - 00:19:51.585
100/368 - 00:19:52.844
200/368 - 00:19:54.138
300/368 - 00:19:55.376
Iteration completed 00:19:56.227, rn=0.07447388425584783
Iteration 1
0/368 - 00:19:56.287
100/368 - 00:19:57.770
200/368 - 00:19:59.119
300/368 - 00:20:00.352
Iteration completed 00:20:01.169, rn=0.07455105983368815
Iteration 2

Iteration completed 00:25:58.088, rn=0.07484344958948261
Iteration 1
0/368 - 00:25:58.146
100/368 - 00:25:59.376
200/368 - 00:26:00.580
300/368 - 00:26:01.840
Iteration completed 00:26:02.727, rn=0.07487049046476611
Iteration 2
0/368 - 00:26:02.785
100/368 - 00:26:04.029
200/368 - 00:26:05.290
300/368 - 00:26:06.610
Iteration completed 00:26:07.450, rn=0.07487412697399395
00:26:07.450
mu0           21.845822
lambda_val    56.354429
delta          5.626874
sigma0        12.194289
alpha         30.000000
halted         0.000002
opt_H_loss     0.004678
F_rn           0.074874
Name: 25, dtype: float64
Epoch:  26
Optimization [1]: Current loss: 0.00467963293587462
Optimization [101]: Current loss: 0.004679298746843803
Optimization terminated successfully.
         Current function value: 0.004679
         Iterations: 86
         Function evaluations: 153
Iteration 0
0/368 - 00:26:46.477
100/368 - 00:26:47.703
200/368 - 00:26:48.924
300/368 - 00:26:50.135
Iteration completed 00:26:50.979, rn

KeyboardInterrupt: 

In [ ]:
r, c = 2, 5

plt.figure(figsize=(16, 6))

for i, param in enumerate(update_params_df.columns):
        
    plt.subplot(r, c, i+1)
    plt.plot(update_params_df[param])
    plt.title(param)
    
plt.subplots_adjust(hspace=0.5, wspace=0.35)
plt.suptitle("Cell cycle parameter updates, 100 updates")

In [ ]:
real_deconv1.plot_heatmaps()

In [ ]:
# To triage: 
#  0. Would it be faster to brute force the replication timing deconvolution? Yes
#  1. When do we update N and B? It seems there are multiple iterations required to find
#     a good N and B... We can initialize this.... this should be much faster now....
#  
#  2. Add gamma1 and gamma2 updates (update gamma2 to be in addition to gamma1)
#       (gamma2plus): gamma1 + gamma2plus = gamma2

# Are the iterations to N, B and F required?
#    it takes 3-5 iterations for N, B, and F to converge, why does this need to be
#    performed for every epoch?
#

# The search space may be too large for the function evaluation calls of the optimizer
# to find H....
# restricting the bounds of the variables appears to help....


In [ ]:


plt.figure(figsize=(11, 3))
plt.subplot(3, 1, 1)
plt.imshow(np.linalg.inv(real_deconv1.N) @ real_deconv1.G, cmap='RdBu_r',
          vmin=0, vmax=2, aspect='auto')
plt.xticks([])

plt.subplot(3, 1, 2)
plt.imshow(np.linalg.inv(real_deconv1.N) @ real_deconv1.G @ 
           np.linalg.inv(real_deconv1.B), cmap='RdBu_r',
          vmin=0, vmax=2, aspect='auto')

plt.subplot(3, 1, 3)
plt.axhline(1, c='black', ls='dotted', lw=1)
plt.plot(np.diag(np.linalg.inv(real_deconv1.B)))
plt.ylim(0, 2)
plt.xlim(0, len(real_deconv1.B))

In [ ]:
plt.figure(figsize=(11, 2))
plt.plot(1./np.diag(real_deconv1.B))

In [ ]:
# The learning of B is converging to some strange places..
# Let's keep track of H, N, B, and F during this learning process
# Then we can see how the fits converge, it's possible we will need to rethink how
# B is updated...
